# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

 <div style="background-color:#f1be3e">

_Fill in your group number **from Brightspace**, names, and student numbers._
    
| Group          | X       |
|----------------|---------|
| Lauren But     | 6241468 |
| Bente van Geen | XXXXXXX |
| Femke Knibbe   | 6180280 |
| Henno Kruis     | XXXXXXX |

#### Imports

In [12]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 2

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 3

<div style="background-color:#f1be3e">

_Write your answer here._

### 1.2 Genetic Algorithm

In [13]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    
    """
    def __init__(self, generations, pop_size, mutation_rate):
        self.generations = generations
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.crossover_rate = 0.8
        
    def calculate_fitness(self, chromosome, tsp_data):
        """Calculates fitness as 1 / Total Distance"""
        total_distance = tsp_data.start_distances[chromosome[0]]
        
        for i in range(len(chromosome) - 1):
            p1, p2 = chromosome[i], chromosome[i+1]
            total_distance += tsp_data.distances[p1][p2]
            
        total_distance += tsp_data.end_distances[chromosome[-1]]
        
        total_distance += len(chromosome)
        
        return 1.0 / total_distance
    
    def selection(self, population, fitnesses):
        """Tournament Selection"""
        k = 3
        participants = random.sample(list(zip(population, fitnesses)), k)
        return max(participants, key=lambda x: x[1])[0]
    
    def crossover(self, parent1, parent2):
        """Ordered Crossover"""
        if random.random() > self.crossover_rate:
            return parent1[:]
        
        size = len(parent1)
        start, end = sorted(random.sample(range(size), 2))
        child = [None] * size
        child[start:end] = parent1[start:end]
        
        p2_ptr = 0
        for i in range(size):
            if child[i] is None:
                while parent2[p2_ptr] in child:
                    p2_ptr += 1
                child[i] = parent2[p2_ptr]
          
        return child
    
    def mutation(self, chromosome):
        """Swap Mutation"""
        if random.random() < self.mutation_rate:
            idx1, idx2 = random.sample(range(len(chromosome)), 2)
            chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
        return chromosome
        
    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        """Main method that runs GA"""
        num_products = len(tsp_data.product_locations)
        population = [random.sample(range(num_products), num_products) for _ in range(self.pop_size)]
        
        best_chromosome = None
        best_fitness = -1
        
        for gen in range(self.generations):
            fitnesses = [self.calculate_fitness(chrom, tsp_data) for chrom in population]
            
            current_best_idx = np.argmax(fitnesses)
            if fitnesses[current_best_idx] > best_fitness:
                best_fitness = fitnesses[current_best_idx]
                best_chromosome = population[current_best_idx][:]
                
            new_population = []
            new_population.append(best_chromosome)
            
            while len(new_population) < self.pop_size:
                p1 = self.selection(population, fitnesses)
                p2 = self.selection(population, fitnesses)
                
                child = self.crossover(p1, p2)
                child = self.mutation(child)
                new_population.append(child)
                
            population = new_population
            
        return best_chromosome

#### Question 4

<div style="background-color:#f1be3e">

The genes represent the products (locations) that the robot must visit. Each gene corresponds to a single product, identified by its index. Our chromosomes will represent our encoded solution, so in this case the complete route that a robot will follow. We will encode the chromosomes by keeping an ordered sequence of these products, so it is clear in which order the robot visits the products. The chromosome is a permutation of all products, ensuring each product is visited exactly once without duplication.

#### Question 5

<div style="background-color:#f1be3e">

We will use a fitness function based on the total length of the route represented by a chromosome. The total distance for each chromosome is computed by summing the distance from the start position to the first product, the distances between consecutive products and the distance from the last product to the end position. The objective is to minimize this total route length since a shorter route corresponds to a more efficient solution. The fitness function can therefore be defined as the inverse of the route length. This is a suitable choice because shorter routes will result in higher fitness values. This directly reflects the objective of finding the shortest possible route. We also add a constant value for each product in the route to account for the time the robot spends picking up items as specified.

#### Question 6

<div style="background-color:#f1be3e">

For the parents we will use Tournament Selection. This means that a small group of individuals is randomly chosen from the population and the one with the highest fitness gets to be a parent. This is suitable in this situation because it balances both exploration and exploitation. In this case the best individual isn't immediately picked, keeping some diversity and ensuring that the robot isn't immediately stuck with a sub-optimal route too early on.

#### Question 7

<div style="background-color:#f1be3e">

We used Crossover and Mutation. Crossover is a reproduction phase, where it takes two parent routes (that were selected on their relatively short path) and then combines their "DNA" to create a child route. This is used in hopes that the best 'qualities' (segments of the path) from the parents are continued onto their child.

Mutation is an innovation phase, where it takes a child route and makes a random, small change to it. THis can be for example swapping two stops in the route.

We chose these operations since we cannot just 'average' two solutions. We need Crossover to further exploit the already relatively good routes that have been found and Mutation to explore new possibilities (and therefore possibly better paths) to prevent the robot from getting stuck in a good route that isn't the absolute shortest.

#### Question 8

<div style="background-color:#f1be3e">

We prevent the algorithm getting stuck in a local minima by relying on Mutation and maintaining population diversity. A local minimum is when the population becomes very similar and it settles into a good route without that being the absolute shortest. To prevent this there are two ways:
- Mutation, which is also explained above (swapping two things in a route), which introduces 'genetic noise', allowing the algorithm to sometimes jump to a completely different part of the search space that may contain a better solution. 
- Tournament Selection: Using a tournament size of k=3 instead of just picking the absolute best parents every time, we get some 'sub-optimal' routes that can reproduce. We hope to include a specific 'perfect' part that we need for the optimal route later on.

#### Question 9

<div style="background-color:#f1be3e">

Elitism is a genetic algorithm strategy, which copies the best-performing individual(s) from the current generation directly into the next generation without any modification by crossover or mutation.

In this case we did apply elitism in the solve_tsp method with line 83 and 84 of the code block.

We did this because if we didn't, the best route found in generation 10 could get destroyed by a random mutation or bad crossover in generation 11 on accident. When using elitism, we can guarantee that the quality of the solution will not decrease, ensuring that the robot will always remember the shortest path that has been found so far. This leads to faster and more stable convergence towards finding the optimal route.

#### Question 10

In [14]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 20
generations = 20
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
tsp_data.write_action_file(solution, "./../data/tsp_solution.txt")

TypeError: GeneticAlgorithm.__init__() missing 1 required positional argument: 'mutation_rate'

<div style="background-color:#f1be3e">

_Put your code extra blocks above (if any) and write your answer here._

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 12

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 13

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

#### Question 14

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

### 2.3 Implementing the Ant Algorithm

In [15]:
# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


In [16]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        pass

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        pass

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        pass

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        pass

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        pass

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [17]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()
        pass

In [18]:
# Please keep your parameters for the ACO easily changeable here
gen = 1
no_gen = 1
q = 1600
evap = 0.1

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))

shortest_route.write_to_file("./../data/hard_solution.txt")

Ready reading maze file ./../data/hard_maze.txt
Time taken: 0.0


AttributeError: 'NoneType' object has no attribute 'size'

### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [ ]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [19]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

Ready reading maze file ./../data/hard_maze.txt


AttributeError: 'NoneType' object has no attribute 'size'

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**